# Chapter 9 — Activation Functions

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 9:** Activation Functions

## Isi Chapter 9

1. Apa itu activation function, dan empat syarat yang baik
2. Activation untuk hidden layer: sigmoid dan tanh
3. Activation untuk output layer: tanpa activation, sigmoid, dan softmax
4. Cara memasang activation dalam forward dan backward propagation
5. Upgrade jaringan MNIST dengan tanh dan softmax

## Catatan data

Bagian 5 memakai dataset MNIST dari `keras.datasets`, seperti Chapter 8. Jalankan
bagian itu di Google Colab. Notebook ini disimpan tanpa output yang sudah dieksekusi.

## 1. Apa itu activation function?

Activation function adalah fungsi yang diterapkan pada setiap neuron di sebuah layer
saat prediksi. Relu yang sudah saya pakai di chapter sebelumnya adalah salah satu
contohnya. Secara sederhana, activation function menerima satu angka dan mengembalikan
angka lain. Tidak semua fungsi cocok untuk ini. Ada empat syarat yang saya catat:

1. **Kontinu dan domainnya tak terbatas**: setiap input harus punya output.
2. **Monotonik**: fungsi selalu naik atau selalu turun, tidak berbalik arah. Ini bukan syarat keras, tetapi jika fungsi berbalik, ada banyak input berbeda yang menghasilkan output sama, dan ini membuat arah koreksi error jadi kabur.
3. **Nonlinear**: fungsi harus melengkung. Fungsi garis lurus hanya menskala sinyal, sedangkan fungsi lengkung memungkinkan satu input mengubah seberapa berkorelasi neuron dengan input lainnya.
4. **Efisien dihitung**: fungsi dan turunannya dipanggil sangat sering, jadi harus cepat.

Saya cek empat syarat ini dengan dua fungsi yang akan dipakai: sigmoid dan tanh.

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return (x > 0) * x

xs = np.array([-5.0, -1.0, 0.0, 1.0, 5.0])
print('x          :', xs)
print('sigmoid(x) :', np.round(sigmoid(xs), 4))
print('tanh(x)    :', np.round(tanh(xs), 4))
print('relu(x)    :', relu(xs))

Sigmoid selalu menghasilkan nilai antara 0 dan 1, sehingga cocok untuk ditafsirkan
sebagai probabilitas. Tanh menghasilkan nilai antara -1 dan 1, sehingga bisa memberi
korelasi negatif. Relu tidak dibatasi di sisi positif, tetapi mematikan semua nilai
negatif.

## 2. Turunan dari output

Dalam backpropagation, saya perlu slope (turunan) dari setiap activation. Cara yang
efisien: turunan bisa dihitung dari output layer saat forward propagation. Tabel yang
saya pakai:

| Fungsi  | Forward                           | Turunan (backprop delta)        |
|---------|-----------------------------------|---------------------------------|
| relu    | `output = input * (input > 0)`    | `output * (output > 0)`         |
| sigmoid | `output = 1 / (1 + exp(-input))`  | `output * (1 - output)`         |
| tanh    | `output = tanh(input)`            | `1 - output**2`                 |

Saya cek apakah rumus turunan ini benar dengan membandingkannya dengan turunan numerik
(selisih kecil dari fungsi asli):

In [ ]:
def sigmoid2deriv(output):
    return output * (1 - output)

def tanh2deriv(output):
    return 1 - (output ** 2)

def relu2deriv(output):
    return output > 0

def numeric_deriv(f, x, eps=1e-6):
    return (f(x + eps) - f(x - eps)) / (2 * eps)

xs = np.array([-2.0, -0.5, 0.5, 2.0])   # hindari 0 untuk relu

print('sigmoid: rumus  ', np.round(sigmoid2deriv(sigmoid(xs)), 6))
print('sigmoid: numerik', np.round(numeric_deriv(sigmoid, xs), 6))
print('tanh   : rumus  ', np.round(tanh2deriv(tanh(xs)), 6))
print('tanh   : numerik', np.round(numeric_deriv(tanh, xs), 6))
print('relu   : rumus  ', relu2deriv(xs).astype(float))
print('relu   : numerik', np.round(numeric_deriv(relu, xs), 6))

Hasilnya sama untuk ketiga fungsi. Artinya, turunan bisa dihitung dari output layer
tanpa perlu input aslinya.

## 3. Activation untuk output layer

Pilihan output layer tergantung pada apa yang diprediksi:

- **Tanpa activation**: untuk prediksi nilai mentah yang tidak dibatasi 0–1, misalnya suhu.
- **Sigmoid**: untuk beberapa probabilitas biner yang tidak saling terkait, misalnya menang atau tidak, cedera atau tidak.
- **Softmax**: untuk memilih satu label dari banyak kemungkinan, misalnya digit MNIST.

Softmax paling sering dipakai untuk klasifikasi satu label. Saya mengecek alasannya
dengan skor mentah (raw dot product) untuk 10 digit, di mana digit 9 mendapat skor
100 dan sisanya 0:

In [ ]:
def softmax(x):
    temp = np.exp(x - np.max(x))     # dikurangi max agar tidak overflow, hasilnya sama
    return temp / np.sum(temp)

raw = np.array([0.0] * 9 + [100.0])   # index 0-8 = 0, index 9 = 100

print('sigmoid :', np.round(sigmoid(raw), 3))
print('softmax :', np.round(softmax(raw), 3), '| jumlah =', round(float(np.sum(softmax(raw))), 6))

Dengan sigmoid, digit 9 memang tertinggi, tetapi setiap digit lain tetap memiliki nilai
sekitar 0.5. Dengan softmax, hampir semua probabilitas berada di digit 9, dan jumlah
seluruhnya sama dengan 1.

Perbedaan ini juga terlihat pada error. Jika targetnya one-hot (digit 9 bernilai 1 dan
sisanya 0), error kuadrat dari sigmoid di sini sekitar 0.25 pada setiap digit lain,
walaupun prediksi untuk 9 sudah benar:

In [ ]:
target = np.array([0.0] * 9 + [1.0])

sig_out = sigmoid(raw)
sm_out = softmax(raw)

sig_error = (target - sig_out) ** 2
sm_error = (target - sm_out) ** 2

print('error sigmoid per digit :', np.round(sig_error, 3))
print('error softmax per digit :', np.round(sm_error, 3))
print('total error sigmoid     :', round(float(np.sum(sig_error)), 3))
print('total error softmax     :', round(float(np.sum(sm_error)), 6))

Dengan sigmoid, jaringan masih mendapat koreksi besar walaupun prediksinya benar,
karena sigmoid juga ingin setiap digit lain bernilai 0. Softmax tidak memaksa itu,
sehingga error-nya hampir nol. Ini penting saat backpropagation, karena digit yang
mirip (misalnya 2 dan 3) tidak dihukum terlalu keras.

## 4. Memasang activation dalam jaringan

Untuk menambahkan activation pada hidden layer, saya mengubah forward propagation:
`layer_1 = activation(np.dot(layer_0, weights_0_1))`. Di backpropagation, saya
mengalikan delta yang datang dari layer berikutnya dengan turunan activation di titik
yang sama, sehingga layer_1_delta menjadi:

```python
layer_1_delta = layer_2_delta.dot(weights_1_2.T) * activation2deriv(layer_1)
```

Turunan ini memastikan node yang tidak berpengaruh pada error tidak ikut diperbarui
banyak. Untuk sigmoid, node yang nilainya sangat positif atau sangat negatif punya
turunan mendekati 0, sehingga bobotnya berubah sedikit.

## 5. Upgrade jaringan MNIST

Sekarang saya pakai tanh di hidden layer dan softmax di output layer. Beberapa hal
harus disesuaikan:

- **Inisialisasi bobot** untuk tanh dibuat lebih sempit (sekitar -0.01 sampai 0.01), karena tanh lebih cocok dengan bobot kecil.
- **Alpha** perlu dibuat jauh lebih besar (2) agar jaringan mencapai skor yang baik dalam 300 iterasi.
- **Error kuadrat** tidak dihitung di sini, karena softmax idealnya dipakai dengan cross-entropy. Fungsi itu belum dibahas di chapter ini, jadi saya hanya mencatat akurasi.

Kode di bawah memakai dataset yang sama dengan Chapter 8. Jalankan di Colab.

In [ ]:
import numpy as np, sys
from keras.datasets import mnist

np.random.seed(1)
(x_train, y_train), (x_test, y_test) = mnist.load_data()

images = x_train[0:1000].reshape(1000, 28 * 28) / 255
labels_raw = y_train[0:1000]
labels = np.zeros((len(labels_raw), 10))
for i, l in enumerate(labels_raw):
    labels[i][l] = 1

test_images = x_test.reshape(len(x_test), 28 * 28) / 255
test_labels = np.zeros((len(y_test), 10))
for i, l in enumerate(y_test):
    test_labels[i][l] = 1

def tanh_fn(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax_fn(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

alpha, iterations, hidden_size = (2, 300, 100)
pixels_per_image, num_labels = (784, 10)
batch_size = 100

weights_0_1 = 0.02 * np.random.random((pixels_per_image, hidden_size)) - 0.01
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))

        layer_0 = images[batch_start:batch_end]
        layer_1 = tanh_fn(np.dot(layer_0, weights_0_1))
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax_fn(np.dot(layer_1, weights_1_2))

        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) ==
                               np.argmax(labels[batch_start+k:batch_start+k+1]))

        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    if j % 10 == 0:
        test_correct_cnt = 0
        for i in range(len(test_images)):
            layer_0 = test_images[i:i+1]
            layer_1 = tanh_fn(np.dot(layer_0, weights_0_1))
            layer_2 = np.dot(layer_1, weights_1_2)
            test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))
        print('I:', j,
              '| Test-Acc:', round(test_correct_cnt / float(len(test_images)), 4),
              '| Train-Acc:', round(correct_cnt / float(len(images)), 3))

Yang saya harapkan dari upgrade ini: akurasi uji lebih tinggi daripada jaringan relu
di Chapter 8, dan akurasi latih naik dengan stabil. Angka pastinya belum saya
verifikasi, karena dataset MNIST tidak tersedia di sandbox. Setelah notebook dijalankan
di Colab, bandingkan angka akhir dengan Chapter 8 untuk melihat apakah pola itu benar.

## Key Ideas — Chapter 9

- Activation function harus kontinu, monotonik (tidak wajib tetapi sangat disarankan), nonlinear, dan efisien dihitung.
- Sigmoid menghasilkan nilai 0 sampai 1 dan cocok untuk probabilitas biner. Tanh menghasilkan nilai -1 sampai 1 dan sering lebih baik di hidden layer.
- Turunan sigmoid, tanh, dan relu bisa dihitung dari output layer saat forward propagation.
- Untuk memilih satu label dari banyak, softmax lebih cocok daripada sigmoid, karena softmax membuat probabilitasnya berjumlah 1 dan tidak memaksa setiap label lain menjadi 0.
- Mengganti activation biasanya memerlukan penyesuaian inisialisasi bobot dan alpha.

## Kesimpulan

Chapter 9 memberi saya cara memilih activation untuk hidden layer dan output layer,
serta cara memasangnya dalam forward dan backward propagation. Untuk hidden layer, tanh
sering lebih baik daripada sigmoid. Untuk klasifikasi satu label, softmax adalah pilihan
yang lebih tepat.